In [1]:
%run healthcare1_msft_config_notebook {"enable_spark_setup" : true, "enable_packages_mount" : false}

StatementMeta(, 71fdd6e6-fd7a-4739-87f2-7af0711b2d08, 7, Finished, Available, Finished)

Spark configuration setup completed


In [17]:
import json
from pyspark.sql import SparkSession
import pyspark.sql.types as T

def load_config_file(spark: SparkSession, config_path: str) -> str:
    """
    Load the entire content of the file specified by config_path using Spark's wholeTextFiles method.

    Args:
        spark (SparkSession): The active Spark session to utilize.
        config_path (str): The path to the configuration file which needs to be read.

    Returns:
        str: The complete content of the file as a string.
    """
    schema_content = spark.sparkContext.wholeTextFiles(config_path).collect()[0][1]
    return str(schema_content)

def load_schema_from_avro_schema(spark: SparkSession, avro_schema: str) -> T.StructType:
    """
    Convert an AVRO schema string to a Spark StructType using internal Spark SQL utilities.

    This function requires the Spark Avro package to be included in the Spark session.

    Args:
        spark (SparkSession): Spark session to access internal schema conversion utilities.
        avro_schema (str): The string representation of the AVRO schema.

    Returns:
        T.StructType: The AVRO schema converted to Spark's StructType.
    """
    java_schema_type = spark.\
        _jvm.org.apache.spark.sql.avro.SchemaConverters.toSqlType(  # type: ignore
        spark._jvm.org.apache.avro.Schema.Parser().parse(avro_schema)  # type: ignore
        )
    json_schema = json.loads(java_schema_type.dataType().json())
    
    return T.StructType.fromJson(json_schema)


def avro_to_empty_delta(src_dir, target_dir, spark):
    # Ensure the target directory exists
    if not mssparkutils.fs.exists(target_dir):
        mssparkutils.fs.mkdirs(target_dir)

    # List all files in the source directory
    files = mssparkutils.fs.ls(src_dir)

    # Perfom action according to type of schema
    # Filter for Avro schema files, assuming they have '.avsc' extension
    avro_files = [file.path for file in files if file.name.endswith(".avsc")]

    for avro_path in avro_files:
        # Read the full Avro schema from the file using the enhanced loading function
        schema_content = load_config_file(spark, avro_path)

        # Convert Avro JSON schema to Spark StructType
        df_schema = load_schema_from_avro_schema(spark, schema_content)

        # Create an empty DataFrame with this schema
        empty_df = spark.createDataFrame([], df_schema)
        #print(f"Created empty DataFrame with schema: {empty_df.schema}")

        # Infer schema and visualize
        empty_df.printSchema()
        empty_df.show()

        # Construct the Delta file path
        delta_path = f"{target_dir}/{avro_path.split('/')[-1].replace('.avsc', '')}"

        # Write the empty DataFrame as a Delta file
        empty_df.coalesce(1).write.format('delta').mode('overwrite').save(delta_path)
    
    print("Conversion process completed.")

StatementMeta(, 71fdd6e6-fd7a-4739-87f2-7af0711b2d08, 23, Finished, Available, Finished)

In [18]:
# Specify the source directory. See the comments for an example of how to place your files.
#source_directory = "abfss://<workspace-name>@msit-onelake.dfs.fabric.microsoft.com/<lakehouse-name>.Lakehouse/Files/schema/source"
source_directory = ""

StatementMeta(, 71fdd6e6-fd7a-4739-87f2-7af0711b2d08, 24, Finished, Available, Finished)

In [19]:
# Specify the target directory
#target_directory = "abfss://<workspace-name>@msit-onelake.dfs.fabric.microsoft.com/<lakehouse-name>.Lakehouse/Files/schema/target"
target_directory = ""

StatementMeta(, 71fdd6e6-fd7a-4739-87f2-7af0711b2d08, 25, Finished, Available, Finished)

In [20]:
# Call the function to process the schemas
load_schema_json(source_directory, target_directory, spark)

StatementMeta(, 71fdd6e6-fd7a-4739-87f2-7af0711b2d08, 26, Finished, Available, Finished)

Target directory already exists: abfss://hds_marvin_poc_create_delta_tables@msit-onelake.dfs.fabric.microsoft.com/healthcare1_msft_bronze.Lakehouse/Files/jsonschema/target
Files in source directory: [FileInfo(path=abfss://hds_marvin_poc_create_delta_tables@msit-onelake.dfs.fabric.microsoft.com/healthcare1_msft_bronze.Lakehouse/Files/jsonschema/source/table_schema.json, name=table_schema.json, size=8127)]
JSON files found: ['abfss://hds_marvin_poc_create_delta_tables@msit-onelake.dfs.fabric.microsoft.com/healthcare1_msft_bronze.Lakehouse/Files/jsonschema/source/table_schema.json']
Processing file: abfss://hds_marvin_poc_create_delta_tables@msit-onelake.dfs.fabric.microsoft.com/healthcare1_msft_bronze.Lakehouse/Files/jsonschema/source/table_schema.json
root
 |-- dicomimagingmetastore: struct (nullable = true)
 |    |-- description: string (nullable = true)
 |    |-- format: string (nullable = true)
 |    |-- friendlyname: string (nullable = true)
 |    |-- schema: struct (nullable = true